# Section 10 lab — Intermediate to difficult PySpark exercises

Run the setup once. Each exercise starts with its own small input data and then builds the solution in visible steps.


## Section setup — run once

Create one Spark session and import the built-in functions and window API.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.master('local[2]')
         .appName('PySpark-Intermediate-Exercises')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Classify customer value

Combine arithmetic, conditional expressions and ordering.


### Step 1 — Create customer measures

Each row contains lifetime revenue and completed-order count.


In [ ]:
customer_value = spark.createDataFrame([
    ('C01', 120000.0, 24), ('C02', 45000.0, 12),
    ('C03', 9000.0, 3), ('C04', 25000.0, 0),
], ['customer_id', 'lifetime_revenue', 'completed_orders'])
customer_value.show()


### Step 2 — Calculate average order value safely

Avoid division when the completed-order count is zero.


In [ ]:
valued = customer_value.withColumn('average_order_value', F.when(F.col('completed_orders') > 0, F.col('lifetime_revenue') / F.col('completed_orders')))


### Step 3 — Assign a value band

Apply the highest-value rule first.


In [ ]:
banded = (valued.withColumn('value_band',
          F.when(F.col('lifetime_revenue') >= 100000, 'PLATINUM')
           .when(F.col('lifetime_revenue') >= 40000, 'GOLD')
           .when(F.col('lifetime_revenue') >= 10000, 'SILVER')
           .otherwise('BRONZE'))
          .orderBy(F.desc('lifetime_revenue')))
banded.show()


### Paired literal SQL solution

Use `CASE` to reproduce the calculated value and band.


In [ ]:
customer_value.createOrReplaceTempView('s10_customer_value')
sql_result = spark.sql("""
SELECT *, CASE WHEN completed_orders > 0 THEN lifetime_revenue / completed_orders END AS average_order_value,
  CASE WHEN lifetime_revenue >= 100000 THEN 'PLATINUM'
       WHEN lifetime_revenue >= 40000 THEN 'GOLD'
       WHEN lifetime_revenue >= 10000 THEN 'SILVER' ELSE 'BRONZE' END AS value_band
FROM s10_customer_value ORDER BY lifetime_revenue DESC
""")
sql_result.show()
assert sql_result.collect() == banded.collect()


## Exercise 2 — Expand basket items

Turn array elements into rows and count item frequency.


### Step 1 — Create baskets with arrays

One basket can contain several product codes.


In [ ]:
baskets = spark.createDataFrame([
    ('B01', 'C01', ['P01', 'P02', 'P01']),
    ('B02', 'C02', ['P02', 'P03']),
    ('B03', 'C01', ['P01']),
], ['basket_id', 'customer_id', 'items'])
baskets.show(truncate=False)


### Step 2 — Explode one array element per row

Keep the basket key beside each item.


In [ ]:
basket_items = baskets.select('basket_id', 'customer_id', F.explode('items').alias('product_id'))
basket_items.orderBy("basket_id", "product_id").show()


### Step 3 — Count product appearances

Repeated items in the same basket remain separate appearances.


In [ ]:
item_frequency = basket_items.groupBy('product_id').count().orderBy(F.desc('count'), 'product_id')
item_frequency.show()
assert item_frequency.first()["product_id"] == "P01"


### Paired literal SQL solution

Use a lateral view to explode each array.


In [ ]:
baskets.createOrReplaceTempView('s10_baskets')
sql_result = spark.sql("""
SELECT product_id, COUNT(*) AS count
FROM s10_baskets
LATERAL VIEW EXPLODE(items) item_rows AS product_id
GROUP BY product_id ORDER BY count DESC, product_id
""")
sql_result.show()
assert sql_result.collect() == item_frequency.collect()


## Exercise 3 — Keep the latest record for each key

Use `row_number` to remove older versions deterministically.


### Step 1 — Create product updates

Some products have more than one update timestamp.


In [ ]:
updates = spark.createDataFrame([
    ('P01', 'Keyboard', 1200.0, '2026-11-01 09:00:00'),
    ('P01', 'Keyboard', 1250.0, '2026-11-02 10:00:00'),
    ('P02', 'Headphones', 2200.0, '2026-11-01 11:00:00'),
    ('P02', 'Headphones Pro', 2400.0, '2026-11-03 08:00:00'),
    ('P03', 'Monitor', 15000.0, '2026-11-02 12:00:00'),
], ['product_id', 'product_name', 'price', 'updated_at']).withColumn('updated_at', F.to_timestamp('updated_at'))
updates.orderBy("product_id", "updated_at").show()


### Step 2 — Number versions newest first

Partition by the business key and order timestamps descending.


In [ ]:
latest_window = Window.partitionBy('product_id').orderBy(F.desc('updated_at'))
numbered = updates.withColumn('version_position', F.row_number().over(latest_window))


### Step 3 — Keep position one

Drop the technical position column from the final result.


In [ ]:
latest_products = numbered.filter(F.col('version_position') == 1).drop('version_position').orderBy('product_id')
latest_products.show()
assert latest_products.count() == 3


### Paired literal SQL solution

Apply the same row-number window in a common table expression.


In [ ]:
updates.createOrReplaceTempView('s10_updates')
sql_result = spark.sql("""
WITH numbered AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY updated_at DESC) AS version_position
  FROM s10_updates
)
SELECT product_id, product_name, price, updated_at
FROM numbered WHERE version_position = 1 ORDER BY product_id
""")
sql_result.show()
assert sql_result.collect() == latest_products.collect()


## Exercise 4 — Compare daily values with rolling windows

Calculate previous-day change and a three-row moving average.


### Step 1 — Create category sales by date

Each category has an independent timeline.


In [ ]:
daily_sales = spark.createDataFrame([
    ('2026-11-01', 'Audio', 1000.0), ('2026-11-02', 'Audio', 1400.0),
    ('2026-11-03', 'Audio', 1200.0), ('2026-11-04', 'Audio', 1800.0),
    ('2026-11-01', 'Electronics', 5000.0), ('2026-11-02', 'Electronics', 4500.0),
    ('2026-11-03', 'Electronics', 6000.0),
], ['sale_date', 'category', 'revenue']).withColumn('sale_date', F.to_date('sale_date'))
daily_sales.orderBy("category", "sale_date").show()


### Step 2 — Define ordered and rolling windows

The rolling frame contains the current row and two earlier rows.


In [ ]:
ordered_days = Window.partitionBy('category').orderBy('sale_date')
three_days = ordered_days.rowsBetween(-2, Window.currentRow)


### Step 3 — Add previous value, change and moving average

The first row in each category has no previous value.


In [ ]:
daily_analysis = (daily_sales
    .withColumn('previous_revenue', F.lag('revenue').over(ordered_days))
    .withColumn('change_from_previous', F.col('revenue') - F.col('previous_revenue'))
    .withColumn('moving_average_3', F.avg('revenue').over(three_days))
    .orderBy('category', 'sale_date'))
daily_analysis.show()


### Paired literal SQL solution

Use `LAG` and an ordered three-row frame.


In [ ]:
daily_sales.createOrReplaceTempView('s10_daily_sales')
sql_result = spark.sql("""
WITH analysed AS (
  SELECT *, LAG(revenue) OVER (PARTITION BY category ORDER BY sale_date) AS previous_revenue,
         AVG(revenue) OVER (PARTITION BY category ORDER BY sale_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_average_3
  FROM s10_daily_sales
)
SELECT sale_date, category, revenue, previous_revenue,
       revenue - previous_revenue AS change_from_previous, moving_average_3
FROM analysed ORDER BY category, sale_date
""")
sql_result.show()
assert sql_result.collect() == daily_analysis.collect()


## Exercise 5 — Calculate category share within each month

Aggregate first, then compare each category with its monthly total.


### Step 1 — Create transaction revenue

Several rows contribute to the same month and category.


In [ ]:
monthly_lines = spark.createDataFrame([
    ('2026-10', 'Audio', 1000.0), ('2026-10', 'Audio', 500.0),
    ('2026-10', 'Electronics', 3500.0), ('2026-10', 'Accessories', 500.0),
    ('2026-11', 'Audio', 1200.0), ('2026-11', 'Electronics', 4800.0),
], ['sale_month', 'category', 'revenue'])
monthly_lines.show()


### Step 2 — Aggregate category revenue

Change the detail to one row per month and category.


In [ ]:
monthly_category = monthly_lines.groupBy('sale_month', 'category').agg(F.sum('revenue').alias('category_revenue'))


### Step 3 — Add monthly total and percentage share

A partition-only window keeps category rows while adding the total.


In [ ]:
month_window = Window.partitionBy('sale_month')
monthly_share = (monthly_category
    .withColumn('month_revenue', F.sum('category_revenue').over(month_window))
    .withColumn('revenue_share_pct', F.round(F.col('category_revenue') / F.col('month_revenue') * 100, 2))
    .orderBy('sale_month', F.desc('category_revenue')))
monthly_share.show()


### Paired literal SQL solution

Aggregate in a CTE, then apply the monthly window total.


In [ ]:
monthly_lines.createOrReplaceTempView('s10_monthly_lines')
sql_result = spark.sql("""
WITH category_totals AS (
  SELECT sale_month, category, SUM(revenue) AS category_revenue
  FROM s10_monthly_lines GROUP BY sale_month, category
), month_totals AS (
  SELECT *, SUM(category_revenue) OVER (PARTITION BY sale_month) AS month_revenue
  FROM category_totals
)
SELECT *, ROUND(category_revenue / month_revenue * 100, 2) AS revenue_share_pct
FROM month_totals ORDER BY sale_month, category_revenue DESC
""")
sql_result.show()
assert sql_result.collect() == monthly_share.collect()


## Exercise 6 — Enrich transactions and identify unknown products

Use a left join for a complete audit and a left-anti join for unmatched rows.


### Step 1 — Create transactions and product references

One transaction deliberately contains an unknown product.


In [ ]:
transactions = spark.createDataFrame([
    ('T01', 'P01', 2), ('T02', 'P02', 1), ('T03', 'P99', 3), ('T04', 'P03', 1),
], ['transaction_id', 'product_id', 'quantity'])
products = spark.createDataFrame([
    ('P01', 'Keyboard', 'Accessories'), ('P02', 'Headphones', 'Audio'), ('P03', 'Monitor', 'Electronics'),
], ['product_id', 'product_name', 'category'])


### Step 2 — Build an auditable left join

A match-status column keeps the unmatched condition visible.


In [ ]:
join_audit = (transactions.join(products, 'product_id', 'left')
    .withColumn('match_status', F.when(F.col('product_name').isNull(), 'UNKNOWN_PRODUCT').otherwise('MATCHED'))
    .orderBy('transaction_id'))
join_audit.show()


### Step 3 — Isolate unknown products with left anti

The anti join returns only unmatched transaction rows.


In [ ]:
unknown_products = transactions.join(products.select('product_id'), 'product_id', 'left_anti').orderBy('transaction_id')
unknown_products.show()
assert unknown_products.count() == 1


### Paired literal SQL solution

Use a left join and `CASE` to reproduce the audit result.


In [ ]:
transactions.createOrReplaceTempView('s10_transactions')
products.createOrReplaceTempView('s10_products')
sql_result = spark.sql("""
SELECT t.product_id, t.transaction_id, t.quantity, p.product_name, p.category,
       CASE WHEN p.product_id IS NULL THEN 'UNKNOWN_PRODUCT' ELSE 'MATCHED' END AS match_status
FROM s10_transactions t LEFT JOIN s10_products p ON t.product_id = p.product_id
ORDER BY t.transaction_id
""")
sql_result.show()
assert sql_result.collect() == join_audit.collect()


## Exercise 7 — Find top products while preserving ties

Aggregate product revenue, then rank inside each category.


### Step 1 — Create product sales

Audio contains a tie at the highest revenue.


In [ ]:
product_sales = spark.createDataFrame([
    ('Audio', 'P01', 3000.0), ('Audio', 'P01', 1000.0),
    ('Audio', 'P02', 4000.0), ('Audio', 'P03', 2500.0),
    ('Electronics', 'P10', 9000.0), ('Electronics', 'P11', 7000.0), ('Electronics', 'P12', 5000.0),
], ['category', 'product_id', 'revenue'])
product_sales.show()


### Step 2 — Aggregate product totals

Create one row per category and product.


In [ ]:
product_totals = product_sales.groupBy('category', 'product_id').agg(F.sum('revenue').alias('revenue'))


### Step 3 — Keep the top two dense ranks

A tie receives the same rank and does not create a gap.


In [ ]:
product_rank = Window.partitionBy('category').orderBy(F.desc('revenue'))
top_products = (product_totals.withColumn('revenue_rank', F.dense_rank().over(product_rank))
                .filter(F.col('revenue_rank') <= 2)
                .orderBy('category', 'revenue_rank', 'product_id'))
top_products.show()


### Paired literal SQL solution

Use `DENSE_RANK` after product-level aggregation.


In [ ]:
product_sales.createOrReplaceTempView('s10_product_sales')
sql_result = spark.sql("""
WITH totals AS (
  SELECT category, product_id, SUM(revenue) AS revenue
  FROM s10_product_sales GROUP BY category, product_id
), ranked AS (
  SELECT *, DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS revenue_rank
  FROM totals
)
SELECT * FROM ranked WHERE revenue_rank <= 2 ORDER BY category, revenue_rank, product_id
""")
sql_result.show()
assert sql_result.collect() == top_products.collect()


## Exercise 8 — Build sessions from event gaps

Start a new session when a user has been inactive for more than thirty minutes.


### Step 1 — Create time-ordered user events

Each user timeline is independent.


In [ ]:
events = spark.createDataFrame([
    ('U01', '2026-11-01 09:00:00', 'view'),
    ('U01', '2026-11-01 09:10:00', 'cart'),
    ('U01', '2026-11-01 10:00:00', 'purchase'),
    ('U02', '2026-11-01 08:00:00', 'view'),
    ('U02', '2026-11-01 08:20:00', 'view'),
    ('U02', '2026-11-01 09:30:00', 'cart'),
], ['user_id', 'event_time', 'event_type']).withColumn('event_time', F.to_timestamp('event_time'))
events.orderBy("user_id", "event_time").show()


### Step 2 — Compare each event with the previous event

Flag the first event and gaps greater than 1,800 seconds.


In [ ]:
event_order = Window.partitionBy('user_id').orderBy('event_time')
with_previous = events.withColumn('previous_time', F.lag('event_time').over(event_order))
flagged = with_previous.withColumn('new_session', F.when(F.col('previous_time').isNull() | ((F.col('event_time').cast('long') - F.col('previous_time').cast('long')) > 1800), 1).otherwise(0))


### Step 3 — Convert flags into session numbers

A cumulative sum creates a stable session ID within each user.


In [ ]:
session_frame = event_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
sessionized = (flagged.withColumn('session_number', F.sum('new_session').over(session_frame))
               .drop('previous_time').orderBy('user_id', 'event_time'))
sessionized.show()


### Paired literal SQL solution

Use `LAG` and a cumulative `SUM` in successive CTEs.


In [ ]:
events.createOrReplaceTempView('s10_events')
sql_result = spark.sql("""
WITH previous_events AS (
  SELECT *, LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS previous_time
  FROM s10_events
), flags AS (
  SELECT *, CASE WHEN previous_time IS NULL OR UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(previous_time) > 1800 THEN 1 ELSE 0 END AS new_session
  FROM previous_events
)
SELECT user_id, event_time, event_type, new_session,
       SUM(new_session) OVER (PARTITION BY user_id ORDER BY event_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_number
FROM flags ORDER BY user_id, event_time
""")
sql_result.show()
assert sql_result.collect() == sessionized.collect()


## Exercise 9 — Score row-level data quality

Evaluate several independent rules and retain the reasons for failure.


### Step 1 — Create valid and invalid transaction rows

The input types are already suitable so the exercise can focus on rules.


In [ ]:
quality_source = spark.createDataFrame([
    ('T01', 'C01', 2, 100.0, '2026-11-01'),
    ('T02', None, 1, 50.0, '2026-11-01'),
    ('T03', 'C03', 0, 75.0, '2026-11-02'),
    ('T04', 'C04', 3, -10.0, None),
], ['transaction_id', 'customer_id', 'quantity', 'unit_price', 'order_date'])
quality_source.show()


### Step 2 — Calculate passed-rule count

There are four rules: customer, quantity, price and date.


In [ ]:
scored = quality_source.withColumn('quality_score',
    F.when(F.col('customer_id').isNotNull(), 1).otherwise(0) +
    F.when(F.col('quantity') > 0, 1).otherwise(0) +
    F.when(F.col('unit_price') >= 0, 1).otherwise(0) +
    F.when(F.col('order_date').isNotNull(), 1).otherwise(0))


### Step 3 — Attach readable failure reasons

Keep all failed rules instead of only the first failure.


In [ ]:
quality_result = (scored.withColumn('reject_reason', F.concat_ws('; ',
    F.when(F.col('customer_id').isNull(), 'missing customer'),
    F.when(F.col('quantity').isNull() | (F.col('quantity') <= 0), 'non-positive quantity'),
    F.when(F.col('unit_price').isNull() | (F.col('unit_price') < 0), 'negative price'),
    F.when(F.col('order_date').isNull(), 'missing date')))
    .orderBy('transaction_id'))
quality_result.show(truncate=False)


### Paired literal SQL solution

Use numeric `CASE` expressions and `CONCAT_WS` for the same score and reasons.


In [ ]:
quality_source.createOrReplaceTempView('s10_quality_source')
sql_result = spark.sql("""
SELECT *,
  CASE WHEN customer_id IS NOT NULL THEN 1 ELSE 0 END
  + CASE WHEN quantity > 0 THEN 1 ELSE 0 END
  + CASE WHEN unit_price >= 0 THEN 1 ELSE 0 END
  + CASE WHEN order_date IS NOT NULL THEN 1 ELSE 0 END AS quality_score,
  CONCAT_WS('; ', CASE WHEN customer_id IS NULL THEN 'missing customer' END,
    CASE WHEN quantity IS NULL OR quantity <= 0 THEN 'non-positive quantity' END,
    CASE WHEN unit_price IS NULL OR unit_price < 0 THEN 'negative price' END,
    CASE WHEN order_date IS NULL THEN 'missing date' END) AS reject_reason
FROM s10_quality_source ORDER BY transaction_id
""")
sql_result.show(truncate=False)
assert sql_result.collect() == quality_result.collect()


## Exercise 10 — Classify inserts, updates, deletes and unchanged rows

Compare an earlier snapshot with a new snapshot using a full outer join.


### Step 1 — Create old and new product snapshots

The examples include every change type.


In [ ]:
old_products = spark.createDataFrame([
    ('P01', 'Keyboard', 1200.0), ('P02', 'Headphones', 2200.0), ('P03', 'Monitor', 15000.0),
], ['product_id', 'product_name', 'price'])
new_products = spark.createDataFrame([
    ('P01', 'Keyboard', 1250.0), ('P02', 'Headphones', 2200.0), ('P04', 'Webcam', 3500.0),
], ['product_id', 'product_name', 'price'])


### Step 2 — Join both snapshots completely

Alias columns so old and new values stay distinguishable.


In [ ]:
snapshot_join = old_products.alias('old').join(new_products.alias('new'), F.col('old.product_id') == F.col('new.product_id'), 'full')


### Step 3 — Assign a change type

Use null-safe equality so two null values are treated as equal.


In [ ]:
changes = (snapshot_join.select(
    F.coalesce(F.col('old.product_id'), F.col('new.product_id')).alias('product_id'),
    F.col('old.product_name').alias('old_name'), F.col('new.product_name').alias('new_name'),
    F.col('old.price').alias('old_price'), F.col('new.price').alias('new_price'),
    F.when(F.col('old.product_id').isNull(), 'INSERT')
     .when(F.col('new.product_id').isNull(), 'DELETE')
     .when(~(F.col('old.product_name').eqNullSafe(F.col('new.product_name')) & F.col('old.price').eqNullSafe(F.col('new.price'))), 'UPDATE')
     .otherwise('UNCHANGED').alias('change_type'))
    .orderBy('product_id'))
changes.show()
assert changes.groupBy('change_type').count().count() == 4


### Paired literal SQL solution

Use a full outer join and null-safe equality with the `<=>` operator.


In [ ]:
old_products.createOrReplaceTempView('s10_old_products')
new_products.createOrReplaceTempView('s10_new_products')
sql_result = spark.sql("""
SELECT COALESCE(old.product_id, new.product_id) AS product_id,
       old.product_name AS old_name, new.product_name AS new_name,
       old.price AS old_price, new.price AS new_price,
       CASE WHEN old.product_id IS NULL THEN 'INSERT'
            WHEN new.product_id IS NULL THEN 'DELETE'
            WHEN NOT (old.product_name <=> new.product_name AND old.price <=> new.price) THEN 'UPDATE'
            ELSE 'UNCHANGED' END AS change_type
FROM s10_old_products old FULL OUTER JOIN s10_new_products new
  ON old.product_id = new.product_id
ORDER BY product_id
""")
sql_result.show()
assert sql_result.collect() == changes.collect()
